In [1]:
import pandas as pd
import json
import requests
import time
import os

In [2]:
url = "https://diavgeia.gov.gr/luminapi/api/search" #Αρχικά χρησιμοποιούμε το επίσημο API της Διάυγειας (luminapi) και στοχεύουμε στο Υπουργείο Μετανάστευσης και Ασύλου χρησιμοποιώντας το μοναδικό του αναγνωριστικό (100056663)
params = {
    "q": 'organizationUid:"100056663"',
    "size": 50, #Ζητάμε 50 αποτελέσματα ανα σελίδα
    "page": 0, # Ξεκινάμε απο τη σελίδα 0
    "sort": "recent", #Ταξινομούμε τα αποτελέσματα με βάση τα πιο πρόδφατα
}

all_decisions = [] #Στη συνέχεια δημιουργούμε μια κενή λίστα (all_decisions), όπου θα αποθηκευτούν προσωρινά όλες οι αποφάσεις

print("Ξεκινάει η συλλογή των αποφάσεων από το API...")
while True: # Επειδή οι αποφάσεις είναι χιλιάδες και δεν μπορούμε να τις πάρουμε όλες μαζί χρησιμοποιούμε ένα βράχο while True
    resp = requests.get(url, params=params, headers={"Accept": "application/json"}, timeout=30) #Σε κάθε επανάληψη, το πρόγραμμα στέλνει αίτημα στο API ζητώνταςα την τρέχουσα σελίδα, ορίζοντας ότι την θέλουμε σε μορφή json
    resp.raise_for_status()
    data = resp.json()

    decisions = data.get("decisions", []) #Αν το API επιστρέφει αποφάσεις, τις αποθηκεύουμε στη συνολική λίστα
    if not decisions:
        break # Μόλις το API επιστρέφει όλες τις αποφάσεις σταματάει τον βράχο

    all_decisions.extend(decisions)
    params["page"] += 1 # Παίρνει μία μία τη κάθε σελίδα
    time.sleep(0.5) #Με την εντολή time.sleep καθυστερούμε το επόμενο αίτημα για να μην μπλοκάρει ο σέρβερ από τα πολλά αιτήματα

print(f" Συλλέχθηκαν συνολικά {len(all_decisions)} αποφάσεις από το API.") # Βλέπουμε πόσα δεδομένα συλλέχθηκαν

raws = [] # Επειδή δεν μας ενδιαφέρουν όλες οι αποφάσεις προσπαθούμε να καθαρίσουμε τα δεδομένα.
raws = [] # Η λίστα όπου θα αποθηκευτούν οι καθαρές αποφάσεις.

lekseis_kleidia = [ 
    "μκο", "μ.κ.ο", "σίτιση", "σιτιση", "στέγαση", "στεγαση",
    "δομή", "δομες", "δομών", "φιλοξεν", "μετανάστ", "μεταναστ",
    "πρόσφυγ", "προσφυγ", "αιτούντες", "ασύλου", "δαπάνη",
    "επιχορήγ", "επιχορηγ", "πληρωμή"
]

apogorevmenes_lekseis = [
    "υπερωρ", "υπερωριακή", "συγκρότηση συνεργείου", "αποζημίωση", 
    "άδεια", "άδειες", "μετάθεση", "απόσπαση", "τοποθέτηση", 
    "κλιμάκιο", "μισθολογ", "υπάλληλος", "υπαλλήλων", "προσωπικού",
    "προμήθεια γραφικής", "έκδοση εγγράφων", "συγκρότηση επιτροπής"
]

print(" Φιλτράρισμα δεδομένων (Θετικό & Αρνητικό)...")
stat_excluded = 0  # Βάζουμε ένα μετρητή για να δούμε πόσα αρχεία κόπηκαν ως άχρηστα

for decision in all_decisions: 
    subject_text = decision.get("subject", "") 
    thema_low = subject_text.lower() 
    if any(leksi in thema_low for leksi in lekseis_kleidia): # Ελέγχουμε αν υπάρχει τουλαχιστον μια «καλή» λέξη
        if any(oxi_leksi in thema_low for oxi_leksi in apogorevmenes_lekseis): #Ελέγχουμε αν έχει και «απαγορευμένη» λέξη
            stat_excluded += 1  # Αν έχει απαγορευμένη λέξη, το καταγράφουμε
            continue            # και το προσπερνάμε 

        # Αν περάσει και τους δύο ελέγχους, μαζεύουμε τα στοιχεία κανονικά
        organization = decision.get("organization") or {} 
        decision_type = decision.get("decisionType") or {}
        categories_list = decision.get("thematicCategories") or []
        thematic_labels = [cat.get("label", "") for cat in categories_list]
        thematic_str = ", ".join(thematic_labels)
        doc_meta = decision.get("documentMeta") or {}

        raw = { 
            "ada": decision.get("ada", ""),
            "protocol": decision.get("protocolNumber", ""),
            "subject": decision.get("subject", ""),
            "issueDate": decision.get("issueDate", ""),
            "publishTimestamp": decision.get("publishTimestamp", ""),
            "documentUrl": decision.get("documentUrl", ""),
            "organization": organization.get("label", ""),
            "decisionType": decision_type.get("label", ""),
            "thematicCategories": thematic_str,
            "unitIds": ", ".join(decision.get("unitIds", [])) if decision.get("unitIds") else "",
            "fileSize": doc_meta.get("fileSize", 0)
        }
        raws.append(raw) 

with open("filtered_decisions.json", "w", encoding="utf-8") as f: #Δημιουργούμε ένα αρχείο για να βάλουμε τα δεδομένα, ορίζουμε τη κωδικοποιήση utf-8 ώστε να υποστηρίζονται σωστά όλοι οι ελληνικοί χαρακτήρες χωρίς να εμφανίζονται ακαταλαβίστικα σύμβολα  
    json.dump(raws, f, indent=4, ensure_ascii=False) # Παίρνουμε τη λίστα raws που περιέχει τις αποφάσεις και γράφει μέσα στο αρχέιο σε μορφή json.To indent=4 το βάζουμε προκειμένου να υπάρχει όμορφη στοιχιση με 4 κενα και το ensure_ascii=False προκειμένου να γράψει τους ελληνικούς χαρακτήρες ως κανονικά γράμματα και όχι ως κωδικοποιημένα σύμβολα

print(f"Φιλτράρισμα Ολοκληρώθηκε")
print(f" Απορρίφθηκαν {stat_excluded} εσωτερικές/διοικητικές αποφάσεις.")
print(f" Κρατήθηκαν {len(raws)} χρήσιμες αποφάσεις στο 'filtered_decisions.json'.")

Ξεκινάει η συλλογή των αποφάσεων από το API...
 Συλλέχθηκαν συνολικά 38153 αποφάσεις από το API.
 Φιλτράρισμα δεδομένων (Θετικό & Αρνητικό)...
Φιλτράρισμα Ολοκληρώθηκε
 Απορρίφθηκαν 2050 εσωτερικές/διοικητικές αποφάσεις.
 Κρατήθηκαν 8225 χρήσιμες αποφάσεις στο 'filtered_decisions.json'.


In [3]:

df = pd.DataFrame(raws) #Μετατρέπουμε τη λίστα raws που περιέχει τις χιλιάδες αποφάσεςι σε dataframe
df["ada"] = df["ada"].fillna("").astype(str).str.strip() #Καθαρίζουμε το βασικό κλειδί της Δια΄υγειας από τα κε΄να
df = df[df["ada"] != ""] # Πετάμε τις γραμμές που δεν έχουν ΑΔΑ
df = df.drop_duplicates(subset=["ada"]).reset_index(drop=True) # Αφαιρούμε τις διπλές γραμμές με βάση το ΑΔΑ
df['clean_date'] = pd.to_datetime(df['issueDate'], format='%d/%m/%Y %H:%M:%S', errors='coerce') #μετρέπουμε τη στύλη issueDate, σε μια νέα στύλη clean date που θα περιέχει πραγματική, ψηφιακή ημερομηνίας
df['year'] = df['clean_date'].dt.year # εξάγουμε το έτος σε ξεχωριστή στύλη
df['month'] = df['clean_date'].dt.month #εξάγουμε το μήνα σε ξεχωριστή στύλη

print(f"Βρέθηκαν {len(df)} μοναδικές αποφάσεις του Υπουργείου Μετανάστευσης για επεξεργασία.")
df.head()

Βρέθηκαν 8225 μοναδικές αποφάσεις του Υπουργείου Μετανάστευσης για επεξεργασία.


,ada,protocol,subject,issueDate,publishTimestamp,documentUrl,organization,decisionType,thematicCategories,unitIds,fileSize,clean_date,year,month
0,9Ι4Δ46ΜΔΨΟ-ΚΤΘ,175259/15-09-2026,Τροποποίηση της εκτελεστικής σύμβασης FM3-35 «...,15/09/2026 03:00:00,15/09/2026 10:09:34,https://diavgeia.gov.gr/doc/9Ι4Δ46ΜΔΨΟ-ΚΤΘ,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΛΟΙΠΕΣ ΑΤΟΜΙΚΕΣ ΔΙΟΙΚΗΤΙΚΕΣ ΠΡΑΞΕΙΣ,ΟΙΚΟΝΟΜΙΚΗ ΖΩΗ,,0,2026-09-15 03:00:00,2026,9
1,9ΨΒΡ46ΜΔΨΟ-ΕΑΛ,17500,Ανάθεση για την τακτική επισκευή και προμήθεια...,15/09/2026 03:00:00,15/09/2026 08:53:53,https://diavgeia.gov.gr/doc/9ΨΒΡ46ΜΔΨΟ-ΕΑΛ,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΑΝΑΘΕΣΗ ΕΡΓΩΝ / ΠΡΟΜΗΘΕΙΩΝ / ΥΠΗΡΕΣΙΩΝ / ΜΕΛΕΤΩΝ,ΟΙΚΟΝΟΜΙΚΕΣ ΚΑΙ ΕΜΠΟΡΙΚΕΣ ΣΥΝΑΛΛΑΓΕΣ,,0,2026-09-15 03:00:00,2026,9
2,9ΠΞΥ46ΜΔΨΟ-ΒΒ0,174996,Τροποποίηση της με αρ. πρωτ. 97545/28.05.2026 ...,15/09/2026 03:00:00,15/09/2026 08:07:38,https://diavgeia.gov.gr/doc/9ΠΞΥ46ΜΔΨΟ-ΒΒ0,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΠΡΑΞΗ ΠΟΥ ΑΦΟΡΑ ΣΕ ΣΥΛΛΟΓΙΚΟ ΟΡΓΑΝΟ - ΕΠΙΤΡΟΠΗ...,ΟΙΚΟΝΟΜΙΚΕΣ ΚΑΙ ΕΜΠΟΡΙΚΕΣ ΣΥΝΑΛΛΑΓΕΣ,,0,2026-09-15 03:00:00,2026,9
3,6ΣΚΩ46ΜΔΨΟ-ΖΣΟ,172681/11.09.2026,5η Τροποποίηση της Πράξης «Επιχορήγηση ΜΚΟ ΦΑΡ...,11/09/2026 03:00:00,14/09/2026 09:06:56,https://diavgeia.gov.gr/doc/6ΣΚΩ46ΜΔΨΟ-ΖΣΟ,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΑΛΛΗ ΠΡΑΞΗ ΑΝΑΠΤΥΞΙΑΚΟΥ ΝΟΜΟΥ,"ΕΥΡΩΠΑΪΚΗ ΈΝΩΣΗ, ΔΗΜΟΣΙΑ ΔΙΟΙΚΗΣΗ",,0,2026-09-11 03:00:00,2026,9
4,ΨΙ3Γ46ΜΔΨΟ-410,173125/11-09-2026,Διαπιστωτική Απόφαση Έγκρισης Χρηματοδότησης Π...,11/09/2026 03:00:00,11/09/2026 13:47:22,https://diavgeia.gov.gr/doc/ΨΙ3Γ46ΜΔΨΟ-410,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΕΓΚΡΙΣΗ ΔΑΠΑΝΗΣ,ΔΗΜΟΣΙΑ ΔΙΟΙΚΗΣΗ,,0,2026-09-11 03:00:00,2026,9


In [4]:
from pathlib import Path 
pdf_dir = Path("MKO_pdfs")# φτιάχνουμε ένα φάκελο στον υπλογιστή για να αποθηκευονται εκει όλα τα pdf
pdf_dir.mkdir(exist_ok=True) #αν ο φάκελος δεν υπάρχει, τον δημιουργει. Αν υπάρχει ληδη συνεχίζει κανονικά χωρις να βγάλει σφάλμσ
print("Φόρτωση του JSON αρχείου σε Pandas DataFrame...")
with open("filtered_decisions.json", "r", encoding="utf-8") as f: #ανοιγει το αρχείο filtered_decisions.json, για να το διαβασει ("r"), διασφαλίζοντας ότι οι ελληνικοί χαρακτήρες θα εμφανίζονται σωστά
    data_json = json.load(f) #φορτώνει όλα τα περιεχόμενα του αρχείου στη μνήμη του προγράμματος
df = pd.DataFrame(data_json) #μετατρέπει όλα τα δεδομένα σε ένα πινακα df για να μπορει να τα επεξεργαστεί πιο εύκολα

print(f" Το DataFrame δημιουργήθηκε επιτυχώς!")
print(f" Συνολικές αποφάσεις για κατέβασμα: {len(df)}")
display(df.head()) #εμφανίζει τις 5 πρώτες γραμμές του πίνακα στη οθόνη 
max_downloads = None  # το none σημαινει ότι θα κατέβουν όλα τα αρχεία

if max_downloads is not None: #ελέγχουμε αν υπάρχει συγκεκριμένο όριο
    df_download = df.head(max_downloads) #αν εχουμε βάλει όριο κρατάει μόνο τις πρωτες γραμμές που έχουμε ορίσει
else:
    df_download = df #κραταμε ολόκληρο το πίνακα για κατέβασμα
print("Ξεκινάει το αυτόματο κατέβασμα των αρχείων PDF...")
downloaded = 0 #πόσα pdf κατεβηκαν επιτυχως
skipped = 0 #πόσα pdf προσπεράστηκαν
for index, row in df_download.iterrows(): #ξεκινάει μια λούπα που εξετάζει μία-μία όλες τις γραμμές του πίνακα
    ada = row["ada"] #παίρνουμε το αδα απο τη τρέχουσα γραμμή
    pdf_url = row["documentUrl"] #παίρνουμε τον url που βρίσκεται στο αρχείο 
    if pd.isna(pdf_url) or not pdf_url: #ε΄λέγχουμε αν ο σύνδεσμος λείπει ή είναι άδειος
        continue
    filename = f"{ada}.pdf" #ονομάζουμε το αρχείο που θα κατεβάσουμε χρησιμοποιώντας το αδα
    target_path = pdf_dir / filename #φτιάχνουμε την τελική διαδρομή
    if target_path.exists() and target_path.stat().st_size > 0: #ελέγχουμε αν το αρχείο υπάρχει ήδη στον υπολογιστή και αν έχει περιεχόμενο (δεν είναι 0 KB)
        skipped += 1 #αυξάνουμε το μετρητή των αρχείων που προσπεράστηκαν
        continue
    try: #ξεκινάει ένα μπλοκ δοκιμής, αν συμβεί κα΄ποιο λα΄θος, το πρόγραμμα δεν θα κρασάρει, αλλά θα συνεχίσει
        resp = requests.get(pdf_url, timeout=30) #κάνουμε αίτηση για να τραβήξει τα pdf, με το timeout=30 λ΄εμε ότι αν το site δεν απαντήσει μεσα σε 30 δευτερόπλεπτα να μην περιμένει και να συνεχίσει
        resp.raise_for_status() #ελέγχουμε αν πέτυχε το κατέβασμα, αν όχι σταματάει τη διαδικασία γι αυτο το αρχείο
        with open(target_path, "wb") as f: #ανοίγουμε ένα νέο, άδειο αρχείο στον υπολογιστή για να γραφτεί το pdf, το wb σημαίνει να το γράψουμε σε δυαδική μορφή 
            f.write(resp.content) # αποθηκεύουμε τα δεδομένα μέσα στο αρχείο
            
        downloaded += 1 # αυξάνουμε τον μετρητη επιτυχημένων λήψεων κατα 1
        print(f" [{downloaded}] Κατέβηκε: {filename}") #βλέπουμε ποιο αρχείο κατέβηκε και πόσα έχουν κατέβει
        time.sleep(0.4) #κάνουμε το προγραμμα να ξεκουραστει για 0.4 δευτερολέπτα πριν πάει στο επόμενο αρχείο
        
    except Exception as e: #αν οτιδήποτε παέι στραβα μέσα στο try, το προγραμμα συνεχίζει κανονικά αντι να κρασάρει
        print(f"Αποτυχία για τον ΑΔΑ {ada}: {e}") #τυπώνει στην οθόνη ποιο αρχείο απέτυχε και το λόγο της αποτυχίας
        time.sleep(1)
print(f" Νέα PDF που κατέβηκαν τώρα: {downloaded}")
print(f" PDF που υπήρχαν ήδη και προσπεράστηκαν: {skipped}")

Φόρτωση του JSON αρχείου σε Pandas DataFrame...
 Το DataFrame δημιουργήθηκε επιτυχώς!
 Συνολικές αποφάσεις για κατέβασμα: 8225


,ada,protocol,subject,issueDate,publishTimestamp,documentUrl,organization,decisionType,thematicCategories,unitIds,fileSize
0,9Ι4Δ46ΜΔΨΟ-ΚΤΘ,175259/15-09-2026,Τροποποίηση της εκτελεστικής σύμβασης FM3-35 «...,15/09/2026 03:00:00,15/09/2026 10:09:34,https://diavgeia.gov.gr/doc/9Ι4Δ46ΜΔΨΟ-ΚΤΘ,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΛΟΙΠΕΣ ΑΤΟΜΙΚΕΣ ΔΙΟΙΚΗΤΙΚΕΣ ΠΡΑΞΕΙΣ,ΟΙΚΟΝΟΜΙΚΗ ΖΩΗ,,0
1,9ΨΒΡ46ΜΔΨΟ-ΕΑΛ,17500,Ανάθεση για την τακτική επισκευή και προμήθεια...,15/09/2026 03:00:00,15/09/2026 08:53:53,https://diavgeia.gov.gr/doc/9ΨΒΡ46ΜΔΨΟ-ΕΑΛ,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΑΝΑΘΕΣΗ ΕΡΓΩΝ / ΠΡΟΜΗΘΕΙΩΝ / ΥΠΗΡΕΣΙΩΝ / ΜΕΛΕΤΩΝ,ΟΙΚΟΝΟΜΙΚΕΣ ΚΑΙ ΕΜΠΟΡΙΚΕΣ ΣΥΝΑΛΛΑΓΕΣ,,0
2,9ΠΞΥ46ΜΔΨΟ-ΒΒ0,174996,Τροποποίηση της με αρ. πρωτ. 97545/28.05.2026 ...,15/09/2026 03:00:00,15/09/2026 08:07:38,https://diavgeia.gov.gr/doc/9ΠΞΥ46ΜΔΨΟ-ΒΒ0,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΠΡΑΞΗ ΠΟΥ ΑΦΟΡΑ ΣΕ ΣΥΛΛΟΓΙΚΟ ΟΡΓΑΝΟ - ΕΠΙΤΡΟΠΗ...,ΟΙΚΟΝΟΜΙΚΕΣ ΚΑΙ ΕΜΠΟΡΙΚΕΣ ΣΥΝΑΛΛΑΓΕΣ,,0
3,6ΣΚΩ46ΜΔΨΟ-ΖΣΟ,172681/11.09.2026,5η Τροποποίηση της Πράξης «Επιχορήγηση ΜΚΟ ΦΑΡ...,11/09/2026 03:00:00,14/09/2026 09:06:56,https://diavgeia.gov.gr/doc/6ΣΚΩ46ΜΔΨΟ-ΖΣΟ,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΑΛΛΗ ΠΡΑΞΗ ΑΝΑΠΤΥΞΙΑΚΟΥ ΝΟΜΟΥ,"ΕΥΡΩΠΑΪΚΗ ΈΝΩΣΗ, ΔΗΜΟΣΙΑ ΔΙΟΙΚΗΣΗ",,0
4,ΨΙ3Γ46ΜΔΨΟ-410,173125/11-09-2026,Διαπιστωτική Απόφαση Έγκρισης Χρηματοδότησης Π...,11/09/2026 03:00:00,11/09/2026 13:47:22,https://diavgeia.gov.gr/doc/ΨΙ3Γ46ΜΔΨΟ-410,ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ,ΕΓΚΡΙΣΗ ΔΑΠΑΝΗΣ,ΔΗΜΟΣΙΑ ΔΙΟΙΚΗΣΗ,,0


Ξεκινάει το αυτόματο κατέβασμα των αρχείων PDF...
 [1] Κατέβηκε: 9Ι4Δ46ΜΔΨΟ-ΚΤΘ.pdf
 [2] Κατέβηκε: 9ΨΒΡ46ΜΔΨΟ-ΕΑΛ.pdf
 [3] Κατέβηκε: 9ΠΞΥ46ΜΔΨΟ-ΒΒ0.pdf
Αποτυχία για τον ΑΔΑ Ε1ΖΒ46ΜΔΨΟ-ΩΚΙ: HTTPConnectionPool(host='static.old.diavgeia.gov.gr', port=80): Max retries exceeded with url: /doc/%CE%951%CE%96%CE%9246%CE%9C%CE%94%CE%A8%CE%9F-%CE%A9%CE%9A%CE%99 (Caused by ConnectTimeoutError(<HTTPConnection(host='static.old.diavgeia.gov.gr', port=80) at 0x2369b84a270>, 'Connection to static.old.diavgeia.gov.gr timed out. (connect timeout=30)'))
Αποτυχία για τον ΑΔΑ Ψ30Υ46ΜΔΨΟ-4ΓΜ: HTTPConnectionPool(host='static.old.diavgeia.gov.gr', port=80): Max retries exceeded with url: /doc/%CE%A830%CE%A546%CE%9C%CE%94%CE%A8%CE%9F-4%CE%93%CE%9C (Caused by ConnectTimeoutError(<HTTPConnection(host='static.old.diavgeia.gov.gr', port=80) at 0x2369b6d3890>, 'Connection to static.old.diavgeia.gov.gr timed out. (connect timeout=30)'))
 Νέα PDF που κατέβηκαν τώρα: 3
 PDF που υπήρχαν ήδη και προσπεράστηκαν: 8220


In [5]:
import pypdf #εισάγουμε μια εξωτερική βιβλιοθήκη προκείμένου να επξεργαστούμε τα pdf
from pathlib import Path #εισάγουμε τη βιβλιοθήκη που είναι υπευθθυνη για τη διαχείρηση των φακέλων

print("Ξεκινάει η μετατροπή όλων των pdf σε κείμενο...")
pdf_dir = Path("MKO_pdfs") # ορίζουμε τον φάκελο που είναι αποθηκευμένα τα pdf που κατεβάσαμε

def extract_text_from_pdf(ada): #δημιουργουμε μια συναρτηση που δέχεται έναν αδα και αναλαμβάνει να διβάσει το αντίστοιχο pdf
    filename = f"{ada}.pdf" #σχηματίζουμε το όνομα του αρείου με βάση το αδα
    pdf_path = pdf_dir / filename #βρισκουμε τη διαδρομή του αρχείου στον υπολογιστή
    if not pdf_path.exists(): #ελέγχουμε αν τπάρχει το αρχείο
        return "" #αν δεν υπάρχει, επιστρέφουμε ένα άδειο κείμενο και στματαει εκεί γι αυτο το αρχείο
    try: #ξεκινέμα νέα δοκιμή, γιατι κάποια pdf μπορεί να έινια κατεστραμένα και να μην διβάζονται
        reader = pypdf.PdfReader(pdf_path) #ανοίγουμε το pdf αρχείο χρησιμοποιώντας το pypdf για να μπορ΄έσει να το δει
        text = "" #δημιουργούμε μια άδεια μεταβλητη όπου εκεί θα ενώνεται το κείμενο από όλες τις σελίδες
        for page in reader.pages: #ξεκιναέι μια λούπα που εξετάζει μία-μία όλες τις σελίδες του συγκεκριμένου pdf
            page_text = page.extract_text() #αντιγράφει το γραμμένο κείμενο που βρίσκεται στη συγκεκριμένη σελίδα
            if page_text: #ελέγχουμε αν η σελίδα έχει όντως κείμενο
                text += page_text + "\n" #προσθέτουμε το κείμενο της σελίδας στη κεντρική μεταβλητη text, βάζοντας και μια αλλα΄γη γραμμής (\n) στο τέλος κάθε σελίδας
        return text #επιστρέφει ολόκληρο το κείμενο του pdf 
    except Exception as e: #αν παρουσιαστηκε κάποιο πρόβλημα κατά την ανάγνωση του αρχείου 
        return "" #επιστρέφουμε ένα άδειο κείμενο, ώστε το πρόγραμμα να μην σταματήσει, αλλά να προχωρήσει στο επόμενο αρχείο

df_download["pdf_text"] = df_download['ada'].apply(extract_text_from_pdf) #παίρνουμε ε΄να- ένα όλους τους αδα,και αποθηκεύει όλο το κείμενο που βγάζει σε μια ολοκαίνουργια στήλη στο πίνακα με το όνομα pdf_text
print("Όλο το περιεχόμενο των PDF μετατράπηκε σε string στη στήλη 'pdf_text'.")
df_download[['ada', 'subject', 'pdf_text']].head() #εμφανίζει τις πρωτες γραμμές του πίνακα, δείχνοντας μόνο τρεις συγκεκριμένες στήλες (το αδα, το θέμα και το κείμενο που μόλις βγάλαμε απο τα pdf) για να βεβαιθούμε ότι όλα πήγαν καλά 

Ξεκινάει η μετατροπή όλων των pdf σε κείμενο...


XRef object at 262773 can not be read, some object may be missing
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
XRef object at 288113 can not be read, some object may be missing
XRef object at 228929 can not be read, some object may be missing
parsing for Object Streams
parsing for Object Streams
parsing for Object Streams
parsing for Object Streams
parsing for Object Streams
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Impossible to decode XFormObject /n5: '/n5'
Impossible to decode XFormObject /n5: '/n5'
Impossible to decode XFormObject /n5: '/n5'
Impossible to decode XFormObject /n5: '/n5'
parsing for Object Streams
parsing for Object Streams
Ignoring wrong pointing object 13 0 (offset 0)


Όλο το περιεχόμενο των PDF μετατράπηκε σε string στη στήλη 'pdf_text'.


,ada,subject,pdf_text
0,9Ι4Δ46ΜΔΨΟ-ΚΤΘ,Τροποποίηση της εκτελεστικής σύμβασης FM3-35 «...,Σελίδα 1 από 4\nΕΛΛΗΝΙΚΗ ΔΗΜΟΚΡΑΤΙΑ\nΥπουργείο...
1,9ΨΒΡ46ΜΔΨΟ-ΕΑΛ,Ανάθεση για την τακτική επισκευή και προμήθεια...,\n \nΣελίδα 1 από 9 \n ...
2,9ΠΞΥ46ΜΔΨΟ-ΒΒ0,Τροποποίηση της με αρ. πρωτ. 97545/28.05.2026 ...,Σελίδα 1 από 6 \n \n \nΘέμα: Τροποποίηση της μ...
3,6ΣΚΩ46ΜΔΨΟ-ΖΣΟ,5η Τροποποίηση της Πράξης «Επιχορήγηση ΜΚΟ ΦΑΡ...,\n \nΤαμείο Ασύλου και Μετανάστευσης και ...
4,ΨΙ3Γ46ΜΔΨΟ-410,Διαπιστωτική Απόφαση Έγκρισης Χρηματοδότησης Π...,\nΓΕΝΙΚΗ ΓΡΑΜΜΑΤΕΙΑ ΥΠΟΔΟΧΗΣ\nΑΙΤΟΥΝΤΩΝ ΑΣΥΛΟ...


In [6]:
!pip install google-generativeai


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
!pip install google-genai pydantic


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os  # Φορτώνουμε το os για να διαβάσουμε μεταβλητές περιβάλλοντος.
import json  # Φορτώνουμε το json για να μετατρέπουμε κείμενο JSON σε Python dictionaries.
from getpass import getpass  # Φορτώνουμε το getpass για να πληκτρολογούμε API key χωρίς να φαίνεται στην οθόνη.
gemini_api_key = os.getenv("GEMINI_API_KEY")  # Προσπαθούμε να διαβάσουμε το Gemini API key από το περιβάλλον.
if gemini_api_key is None or gemini_api_key.strip() == "":  # Ελέγχουμε αν δεν βρέθηκε API key.
    gemini_api_key = getpass("Δώσε GEMINI_API_KEY: ")  # Ζητάμε από τον χρήστη να πληκτρολογήσει το API key.
print("Το Gemini API key είναι διαθέσιμο για αυτό το notebook.")  # Τυπώνουμε επιβεβαίωση χωρίς να εμφανίσουμε το ίδιο το key.


Δώσε GEMINI_API_KEY:  ········


Το Gemini API key είναι διαθέσιμο για αυτό το notebook.


In [9]:
import os  
import json  
import pandas as pd  
from pathlib import Path  

gemini_limit = None  # Ορίζουμε το πλήθος των αιτημάτων που θα επεξεργαστει το gemini, στη προκειμένει περίπτωση όλα αφου βάλαμε None
skip_existing_gemini = True #Αν κάποια δεδομένα έχουν υποβληθεί στο gemini για επεξεργασία,δεν θα υποβληθούν σε επεξεργασία ξανα, αλλά θα παραλειφθούν
retry_failed_gemini = False #Αν θα ξαναγίνει προσπάθεια για όσα αιτήματα απέτυχαν 
data_dir = Path(".")  # 

gemini_model = "gemini-2.5-flash"  #Επιλέγουμε το συγκεκριμένο μοντέλο της Google για την επεξεργασία
gemini_url = f"https://generativelanguage.googleapis.com/v1beta/models/{gemini_model}:generateContent" #Κατασκευάζουμε το url στο οοίο θα γίνει το HTTP αίτημα, ενσωματώνοντας το αίτημα του μοντέλου 
gemini_headers = {"Content-Type": "application/json", "x-goog-api-key": gemini_api_key}  #Ορίζουμε ότι τα δεδομένα είναι σε μορφή json και πέρνωντας το API key  για την ταυτοποίηση
gemini_max_chars = 18000 # Θέτουμε ένα συγκεκριμένο όριο χαρακτήρων  

response_schema = {  
    "type": "OBJECT",  
    "properties": {  
        "amount": {"type": "STRING"},  
        "locations": {"type": "STRING"},  
        "why": {"type": "STRING"},  
        "from_whom": {"type": "STRING"},  
        "into_what": {"type": "STRING"},  
        "afm": {"type": "STRING"}  
    },
    "required": ["amount", "locations", "why", "from_whom", "into_what"]  #Ορίζουμε ποια απο τα παραπάνω πεδία είναι υποχρεωτικά 
}

step5_csv = data_dir / "mko_analysis_results_gemini.csv"  #Στο αρχείο αυτο θα αποθηκευτούν τα τελικά αποτελέσματα της ανάλυσης
step5_errors_csv = data_dir / "mko_gemini_errors.csv"  # Στο αρχείο αυτό θα απποθηκευτούν τα τυχον σφάλματα που θα γίνουν
step5_processed_csv = data_dir / "mko_processed_log.csv"  # Στο αρχείο αυτό θα αποθηκευθεί το ιστορικό των εγγραφών που έχουν ήδη επεξεργαστεί

existing_works_df = pd.read_csv(step5_csv) if step5_csv.exists() else None  #Ελέγχουμε αν υπάρχει ήδη το αρχείο αποτελεσμάτων, αν υπάρχει το διαβάζει σε df αλλιώς επιστρέφει None
existing_gemini_processed_df = pd.read_csv(step5_processed_csv) if step5_processed_csv.exists() else None  #Ελέγχουμε αν υπάρχει το αρχείο των επεξεργασμένων εγγραφών, αν δεν υπάρχει επιστρέφει None

if existing_gemini_processed_df is not None and "ada" in existing_gemini_processed_df.columns:  #Ελέγχουμε αν το df των επεξεργασμένων υπάρχει και αν υπάρχει μια στήλη με όνομα αδα
    existing_gemini_processed_df["ada"] = existing_gemini_processed_df["ada"].fillna("").astype(str).str.strip()  #Αν υπάρχει, εφαρμόζουμε καθαρισμό 

already_gemini_done = set()  

if skip_existing_gemini and existing_gemini_processed_df is not None and "ada" in existing_gemini_processed_df.columns:  
    processed_status = existing_gemini_processed_df["status"].fillna("").astype(str)  
    if retry_failed_gemini:  
        already_gemini_done = set(existing_gemini_processed_df.loc[processed_status.isin(["success", "no_findings"]), "ada"])  
    else:  
        already_gemini_done = set(existing_gemini_processed_df["ada"])  
    print(f"Θα παραλείψουμε {len(already_gemini_done)} αποφάσεις που έχουν ήδη περάσει από Gemini.")  

gemini_input_df = df.copy()  #Δημιουργούμε ένα αντίγραφο του αρχικού df για να μην αλλοιωθούν τα αρχικά δεδομένα
gemini_input_df = gemini_input_df[gemini_input_df["pdf_text"].fillna("").str.len() > 0] #  Φιλτράρει το df κρατώντας μόνο τις εγγραφές που έχουν πραγματικό περιεχόμενο στη στήλη pdf_text
gemini_input_df = gemini_input_df[~gemini_input_df["ada"].isin(already_gemini_done)]  #Αφαιρεί όλες τις αποφάσεις των οποίων το αδα βρίσκεται μέσα στο σύνολο already_gemini_done

if gemini_limit is not None:  
    gemini_input_df = gemini_input_df.head(gemini_limit) #Αν έχουμε θέσει ένα ανώτατο όριο αποφάσεων κρατει μονο τις πρωτες γραμμές

print(f"Θα στείλουμε στο Gemini {len(gemini_input_df)} νέες αποφάσεις.")  
print(f"Μοντέλο: {gemini_model}")

Θα παραλείψουμε 8017 αποφάσεις που έχουν ήδη περάσει από Gemini.
Θα στείλουμε στο Gemini 200 νέες αποφάσεις.
Μοντέλο: gemini-2.5-flash


In [25]:
import time
import requests

gemini_rows = []  # Δημιουργούμε μια κενή λίστα που θα χρησιμοποιηθεί μέσα στο loop για να αποθηκευτούν τα επιτυχημένα αποτελέσμτα
gemini_processed_rows = []  #Δημιουργούμε μια κενή λίστα που θα χρησιμοποιηθεί μέσα στο loop για να αποθηκευτούν τα logs επεξεργασίας
gemini_errors = [] #Δημιουργούμε μια κενή λίστα που θα χρησιμοποιηθεί μέσα στο loop για να αποθηκευτούν τα σφάλματα 

for index, row in gemini_input_df.iterrows():  #Ξεκινάει να διαβάζει γραμμή-γραμμή το φιλτραρισμένο df
    ada = str(row.get("ada", ""))  #Παίρνουμε την τιμή της στήλης αδα για την τρέχουσα γραμμή και τη μετρέπεί σε string. Αν δεν υπάρχει βάζει κενό κείμενο
    pdf_text = str(row.get("pdf_text", ""))  #Παίρνουμε το κείμενο της απόφασης (pdf_text) που θα σταλθεί στο gemini
    pdf_text_for_gemini = pdf_text[:gemini_max_chars] #Κρατάμε μόνο τους πρώτους χαρακτήρες του κειμένου μεχρι το όριο που έχουμε ορίσει αρχικα (18.000) 
    
    prompt = f"""
Διάβασε το παρακάτω κείμενο από την απόφαση και εξήγαγε τις εξής πληροφορίες:
1. amount: Ποια είναι τα χρηματικά ποσά που εγκρίθηκαν ή αναφέρονται;
2. locations: Σε ποιες περιοχές, πόλεις ή δομές θα γίνουν οι δράσεις;
3. why: Ποιος είναι ο λόγος / σκοπός της χρηματοδότησης; (Γιατί δίνονται;)
4. from_whom: Ποιος Φορέας, Υπουργείο ή Αρχή εγκρίνει και δίνει τα λεφτά;
5. into_what: Σε τι δράση/έργο πηγαίνουν τα λεφτά ή ποιο είναι το όνομα της ΜΚΟ που τα παίρνει;
6. afm: Ποιο είναι το ΑΦΜ της οργάνωσης (αν υπάρχει γραμμένο);

ΚΕΙΜΕΝΟ PDF:
{pdf_text_for_gemini}
""".strip()

    payload = {  #Κατασκευάζουμε ένα λεξικό που [εριέχει όλες τις απαραίτητες πληροφορίες και ρυθμίσεις που απαιτεί το API του Gemini για να δουλέψει
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "temperature": 0,  
            "response_mime_type": "application/json",
            "response_schema": response_schema  
        }
    }
    
    print("---")
    print(f"Στέλνουμε στο Gemini: {ada}, χαρακτήρες={len(pdf_text_for_gemini)}")
    
    try:
        response = requests.post(gemini_url, headers=gemini_headers, json=payload, timeout=120) #Πραγματοποιούμε τη τη κλήση στο διακομιστή της Google.Στέλνει το payload, τις απαραίτητες κεφαλίδες ασφαλείας (gemini_headers) και ορίζουμε ένα ανώτατο όριο αναμονής 2 λεπτών ώστε να μην κολλήσει το πρόγραμμα αν το API αργήσει να απαντήσει
        print(f"Gemini status_code={response.status_code}") #Τυπώνει τον κωδικό κατάστασης της απάντησης
        
        if response.status_code != 200: #Ελέγχουμε αν η απάντηση δεν είναι επιτυχής
            gemini_errors.append({"ada": ada, "error": response.text[:1000]}) #Αν το api επίστρεψε σφάλμα αποθηκεύει το αδα της απόφασης μαζί με τους πρώτους 1000 χαρακτήρες του μηνύματος σφάλματος στη λίστα gemini_errors για μετέπειτα έλεγχο
            gemini_processed_rows.append({"ada": ada, "status": "error"}) #Αν το api επέστρεψε σφάλμα, καταγράφει στο log επεξεργασίας ότι αυτή η απόφαση απέτυχε με κατάσταση errpr
            print(response.text[:1000])
            continue
            
        response_data = response.json() #Μετατρέπουμε τη απά΄ντηση που ήρθε απο το gemini σε μοφή json
        response_text = response_data["candidates"][0]["content"]["parts"][0]["text"] # Πλογούμαστε μέσα στη δομή της απάντησης του gemini για να απομονώσει το κείμενο που επιστρέφει το μον΄τελο
        parsed_json = json.loads(response_text) #Επείδη το gemini επιδτρέφει κείμενο που μοιάζει με json, η json.load παίρνει αυτό το κείμενο και το μετατρέπει σε αντικέιμενο python, επιτρεπον΄τας μας να διαβάσουμε τα πεδία του
        
        print(" Το Gemini απάντησε επιτυχώς!")
        gemini_processed_rows.append({"ada": ada, "status": "success"}) #Βλ΄επόυμε ποιες οπάφασεις με βαση το αδα επεξεργάστηκαν με επιτυχία
        
        gemini_rows.append({ #Προσθέτουμε ένα νεό λεξικό στη κεντρική λίστα, όπου κάθε κλειδί αντιστοιχεί σε μια πληροφορία που εξήγαγε το gemini
            "ada": ada,
            "title": row.get("subject", ""),
            "amount": parsed_json.get("amount", ""),
            "locations": parsed_json.get("locations", ""),
            "why": parsed_json.get("why", ""),
            "from_whom": parsed_json.get("from_whom", ""),
            "into_what": parsed_json.get("into_what", ""),
            "afm": parsed_json.get("afm", "")
        })
        
        time.sleep(1)  #Σταματάμε την εκτέλεση του κωδικα για 1 δευτερόλεπτο 
        
    except Exception as error: #Αν οποιαδήποτε εντολή του κωδικα αποτύχει, ο κωδικας δεν κρασάρει
        gemini_errors.append({"ada": ada, "error": f"{type(error).__name__}: {error}"}) # Καταγράφουμε το σφάλμα σε μια ειδική λίστα λαθών, αποθηκεύοντας τον τύπο του λάθους και το μήνυμα του μαζί με το αδα 
        gemini_processed_rows.append({"ada": ada, "status": "error"}) #Ενημερώνουμε τη λίστα του ιστορικού ότι η συγκεκριμένη απόφαση απέτυχε
        print(f"ΑΠΟΤΥΧΙΑ: {type(error).__name__}: {error}") #Τυπώνει την αποτυχία και την αιτία της

---
Στέλνουμε στο Gemini: 9Ι4Δ46ΜΔΨΟ-ΚΤΘ, χαρακτήρες=9165
Gemini status_code=429
{
  "error": {
    "code": 429,
    "message": "Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ",
    "status": "RESOURCE_EXHAUSTED"
  }
}

---
Στέλνουμε στο Gemini: 9ΨΒΡ46ΜΔΨΟ-ΕΑΛ, χαρακτήρες=18000
Gemini status_code=429
{
  "error": {
    "code": 429,
    "message": "Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ",
    "status": "RESOURCE_EXHAUSTED"
  }
}

---
Στέλνουμε στο Gemini: 9ΠΞΥ46ΜΔΨΟ-ΒΒ0, χαρακτήρες=16260
Gemini status_code=429
{
  "error": {
    "code": 429,
    "message": "Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Lear

KeyboardInterrupt: 

In [11]:
new_works_df = pd.DataFrame(gemini_rows)  # Μετατρέπουμε τα νέα στοιχεία των ΜΚΟ σε πίνακα.
new_gemini_processed_df = pd.DataFrame(gemini_processed_rows)  # Μετατρέπουμε το νέο processed log σε πίνακα.

works_df = pd.concat([existing_works_df, new_works_df], ignore_index=True) if existing_works_df is not None else new_works_df.copy()
gemini_processed_df = pd.concat([existing_gemini_processed_df, new_gemini_processed_df], ignore_index=True) if existing_gemini_processed_df is not None else new_gemini_processed_df.copy() #Εν΄ώνειι τον πίνακα με το παλι΄ό ιστορικό επξεργασίας που υπήρχε ήδη στον δίσκο, μαζι με το νεό ιστορικό (new_gemini_processed_df) 

if len(gemini_processed_df) > 0 and "ada" in gemini_processed_df.columns: #Ελέγχουμε αν ο πίνακας δεν είναι άδειος και περιέχει όντως τη στήλη αδα, και στη συνέχεια προχωράςι στο επόμενο βήμα
    gemini_processed_df = gemini_processed_df.drop_duplicates(subset=["ada"], keep="last").reset_index(drop=True) #Αν μια απόφαση αδα έχει υποβληθεί δυο φορές, αφαιρεί τα διπλότυπα κρατώνταςα την τελευταία χρονικά καταγραφη (keep=last). Στη συνέχεια ξαναχτίζουμε καθαρεί αρύθμιση για τις γραμμές του πίνακα.

works_df.to_csv(step5_csv, index=False, encoding="utf-8-sig") #Εξάγουμε το πίνακα με όλα τα δεδομένα των ΜΚΟ που επέστρεψε το Gemini σε αρχείο μορφής csv
gemini_processed_df.to_csv(step5_processed_csv, index=False, encoding="utf-8-sig") #Αποθηκεύουμε τον πίνακα του ιστορικού/log, ώστε σε περίπτωση που επαναλάβουμε τη διαδικασία να ξέρει ποιες αποφάσεις έχουν ήδη γίνει 

print(f"Νέες αποφάσεις που αναλύθηκαν τώρα: {len(new_works_df)}")
print(f" Συνολικές αποφάσεις αποθηκευμένες στο αρχείο: {len(works_df)}")

if gemini_errors: #ελέγχουμε αν κατά τη διάρκαις των κλήσεων στο API μπήκαν καταγραφές στη λίστα των λαθών
    new_gemini_errors_df = pd.DataFrame(gemini_errors) #Μετατρέπουμε τα τρέχοντα σφάλματα σε π΄ινακα pandas
    if step5_errors_csv.exists() and step5_errors_csv.stat().st_size > 4: #Ελέγχουμε αν υπάρχει ήδη παλιό αρχείο σφαλμάτων και αν αυτό δεν είναι ίδιο. 
        old_gemini_errors_df = pd.read_csv(step5_errors_csv)
        gemini_errors_df = pd.concat([old_gemini_errors_df, new_gemini_errors_df], ignore_index=True) #Αν ναι διαβάζει τα παλιά λα΄θη και τα ενώνει με τα καινούργια χρησιμοποι΄ώντας τη pd.concat
    else: 
        gemini_errors_df = new_gemini_errors_df
    gemini_errors_df.to_csv(step5_errors_csv, index=False, encoding="utf-8-sig")
    print(f"Σφάλματα Gemini συνολικά: {len(gemini_errors_df)}")
    print(f"Διαδρομή αρχείου λαθών: {step5_errors_csv.resolve()}")
else:
    print(" Κανένα σφάλμα! Όλα κατέβηκαν τέλεια.")

print(f" Το τελικό σου αρχείο αποθηκεύτηκε εδώ: {step5_csv.resolve()}")

# Εμφανίζουμε τις πρώτες 20 γραμμές του πίνακα για να τις δεις στην οθόνη σου
works_df.head(20)

Νέες αποφάσεις που αναλύθηκαν τώρα: 0
 Συνολικές αποφάσεις αποθηκευμένες στο αρχείο: 1527
Σφάλματα Gemini συνολικά: 13068
Διαδρομή αρχείου λαθών: C:\Users\30694\mko_gemini_errors.csv
 Το τελικό σου αρχείο αποθηκεύτηκε εδώ: C:\Users\30694\mko_analysis_results_gemini.csv


,ada,title,amount,locations,why,from_whom,into_what,afm
0,6Σ5Η46ΜΔΨΟ-ΖΝ4,Απόφαση συγκρότησης συνεργείων εργασίας με ενα...,Δεν αναφέρεται,Υπηρεσία Υποδοχής και Ταυτοποίησης της Γενικής...,Συγκρότηση συνεργείων εργασίας με εναλλαγή βαρ...,Ο Προϊστάμενος της Γενικής Διεύθυνσης Διοικητι...,Συγκρότηση συνεργείων εργασίας με εναλλαγή βαρ...,Δεν αναφέρεται
1,Ψ6Γ746ΜΔΨΟ-ΝΒΛ,4η Τροποποίηση της Πράξης «Επιχορήγηση ΜΚΟ ΦΑΡ...,"1.203.670,00€",Αττική,"Βελτίωση της ποιότητας ζωής, 24ωρη φροντίδα κα...","Υπουργείο Μετανάστευσης & Ασύλου, Γενική Γραμμ...",ΜΚΟ ΦΑΡΟΣ ΕΛΠΙΔΑΣ για τη λειτουργία του Κέντρο...,NaN
2,ΨΕΜΠ46ΜΔΨΟ-ΚΛΛ,Ανάκληση διορισμού υπαλλήλου Πανεπιστημιακής Ε...,Δεν αναφέρεται χρηματικό ποσό.,"Αγ. Ιωάννης Ρέντης (έδρα Υπουργείου), Γενική Γ...",Ανάκληση διορισμού υπαλλήλου λόγω μη αποδοχής ...,"Υπουργείο Μετανάστευσης και Ασύλου, μέσω του Π...",Η απόφαση αφορά την ανάκληση διορισμού του Παπ...,Δεν αναφέρεται.
3,ΡΥΔΔ46ΜΔΨΟ-56Φ,ΠΡΟΣΚΛΗΣΗ \n\nΓΙΑ ΤΗΝ ΥΠΟΒΟΛΗ ΠΡΟΤΑΣΗΣ \nΣΤΟ «...,"5.000.000,00€",Ασφαλείς Ζώνες σε Κλειστές Ελεγχόμενες Δομές (...,Για την ενίσχυση του συστήματος προστασίας ασυ...,Υπουργείο Μετανάστευσης και Ασύλου (μέσω της Ε...,Γενική Γραμματεία Ευάλωτων Πολιτών και Θεσμική...,Δεν αναφέρεται
4,ΨΩ9646ΜΔΨΟ-ΣΞ0,7η Τροποποίηση της Πράξης «Παροχή Υπηρεσιών Σί...,"117.605.539,91 €","ΚΕΔ Νήσων, ΚΥΤ Φυλακίου (Βόρεια Ελλάδα), ΠΕ Λέ...","Χρηματοδότηση υπηρεσιών παρασκευής, μεταφοράς ...",Υπουργείο Μετανάστευσης και Ασύλου (Γενική Γρα...,ΥΠΗΡΕΣΙΑ ΥΠΟΔΟΧΗΣ ΚΑΙ ΤΑΥΤΟΠΟΙΗΣΗΣ για την Πρά...,Δεν αναφέρεται
5,ΨΝΝΔ46ΜΔΨΟ-8ΞΑ,"Ανάληψη και Δέσμευση πίστωσης 18.600,00 € από ...","18.600,00 €",Υπουργείο Μετανάστευσης και Ασύλου,Για την προμήθεια υπηρεσιών φωτογραφικής κάλυψ...,Υπουργείο Μετανάστευσης και Ασύλου,Προμήθεια υπηρεσιών φωτογραφικής κάλυψης,NaN
6,97ΧΞ46ΜΔΨΟ-ΣΞΣ,"Ανάληψη και Δέσμευση πίστωσης 4.500,00 € από τ...","4.500,00 €",στο εξωτερικό,για την κάλυψη εξόδων μετακίνησης του Γενικού ...,Υπουργείο Μετανάστευσης και Ασύλου,έξοδα μετακίνησης του Γενικού Γραμματέα Ευάλωτ...,NaN
7,9ΝΒΦ46ΜΔΨΟ-Α0Ν,"Ανάληψη και Δέσμευση πίστωσης 1.000,00 € από τ...","1.000,00 €",στο εσωτερικό και στο εξωτερικό,για την κάλυψη εξόδων κίνησης των Συνεργατών τ...,Υπουργείο Μετανάστευσης και Ασύλου,Έξοδα κίνησης προσωπικού (Συνεργάτες του Γενικ...,NaN
8,9ΖΦ946ΜΔΨΟ-16Ν,"Ανάληψη και Δέσμευση πίστωσης 3.000,00 € από τ...","3.000,00 €",στο εσωτερικό,για την κάλυψη εξόδων μετακίνησης του Γενικού ...,Υπουργείο Μετανάστευσης και Ασύλου,έξοδα μετακίνησης του Γενικού Γραμματέα Ευάλωτ...,NaN
9,6ΜΟ546ΜΔΨΟ-9ΤΤ,"Ανάληψη και Δέσμευση πίστωσης 1.000,00 € από τ...","1.000,00 €",στο εσωτερικό και στο εξωτερικό,για την κάλυψη εξόδων διανυκτέρευσης των Συνερ...,Υπουργείο Μετανάστευσης και Ασύλου,έξοδα διανυκτέρευσης των Συνεργατών του Γενικο...,NaN


In [12]:
import pandas as pd
df_analysis = pd.read_csv("mko_analysis_results_gemini.csv") #Διαβάζουμε το αρχείο δεδομένωνσε μορφή csv και το φορτωνουμε στη μνλημη ως έναν πίνακ που το ονομάζουμε df_analysis
df_analysis['amount_clean'] = df_analysis['amount'].astype(str).str.replace('€', '', regex=False) # Παίρνουμε την αρχική στήλη των ποσών amount, τη μετατρέπουμε σε κείμενο και αφαιρούμε το συμβολο του ευρώ. Το αποτέλεσμα το αποθηκευουμε σε μια νέα στηλη με όνομα amount_clean 
df_analysis['amount_clean'] = df_analysis['amount_clean'].str.replace('.', '', regex=False) #Αφαιρούμε τις τελείες που χρησιμοποιιούνται ως διαχωριστικά χιλιάδων, ωστε να μην μπερδευτεί η Python κατα τη μετατροπή σε αριθμό
df_analysis['amount_clean'] = df_analysis['amount_clean'].str.replace(',', '.', regex=False) #Αντικαθιστούμε το κόμμα της υποδιαστολής με τελεια 
df_analysis['amount_clean'] = df_analysis['amount_clean'].str.extract(r'(\d+\.?\d*)')[0]# Χρησιμοποιούμε regex για να εξάγουμε αποκλειστικά τον αριθμό μέσα από το κείμενο
df_analysis['amount_clean'] = pd.to_numeric(df_analysis['amount_clean'], errors='coerce') #Μετατρέπουμε το κείμενο σε πραγματικό δεκαδικό αριθμό. Το errors=coerce σημαινει ότι αν σε κάποια γραμμλη αποτύχει η μετατροπή, θα τη μετατρέψει σε NaN αντί να κρασάρει το πρόγραμμα

print(f" Η βάση φορτώθηκε επιτυχώς! Έχουμε {len(df_analysis)} αποφάσεις έτοιμες για ερωτήσεις.")

 Η βάση φορτώθηκε επιτυχώς! Έχουμε 1527 αποφάσεις έτοιμες για ερωτήσεις.


In [27]:
import re
import pandas as pd

def extract_amount(text): # Παιρνουμε ένα κείμενο και πρασπαθούμε να εντοπίσουμε και να καθαρίσουμε τα χρηματικα ποσά 
    if pd.isna(text): return 0.0 #Αν το κείμενο είναι κενό επιστρέφουμς 0.0
    match = re.search(r'([\d\.,\s]+)\s*(?:€|ευρώ|euro)', str(text), re.IGNORECASE) #Ψαχνουμε στο κειμενο ένα συνδυασμο ψηφίων, τελειών, κομμάτων και κενών, ο οποίος ακολουθείται απο το σύμβολο του εύρωή τη λε΄ξη ευρώ 
    if match: #Αν βρεθεί τέτοι μοτίβο, εκτελούμε τα παρακάτω
        amount_str = match.group(1).strip() #Απομονώνουμε το κομμάτι που είναι αριθμός και αφαιρούμε τυχόν κενά
        if '.' in amount_str and ',' in amount_str: #Αν ο αριθμός περιέχει και τελεία και κλομμα 
            amount_str = amount_str.replace('.', '').replace(',', '.') #Αφαιρούμε τις τελείες και μετατρέπουμε το κόμμα σε τελεία για να μπορ΄έσει να το διαβάσει η python ως δεκαδικό
        elif ',' in amount_str and '.' not in amount_str: # Αν περιέχει μόνο κόμμα
            amount_str = amount_str.replace(',', '.') #Μετατρέπουμε τοκόμμα σε τελεία
        amount_str = amount_str.replace(' ', '') #Αφαιρεί τυχόν εσωτερικά κενά που μπορέι να έμειναν 
        try: return float(amount_str) #Μετατρέπουμε το τελικό κείμενο σε δεκαδικό αριθμό 
        except: return 0.0 #Αν κάτι πάει στραβα στη μετατροπή, επιστρέφει 0.0 για να μη κρασάρει το πρόγραμμα
    return 0.0 #Αν δεν βρεθεί καθόλου το μοτίβο του ευρώ εξ' αρχής, επιστρέφει 0.0

def extract_beneficiary(text): #Παίρνουμε το κείμενο και προσπα΄θούμε να εντοπίσουμε ποιος είναι ο δικαιούχος, ψάχνοντας για συγκεκριμένες λέξεις- κλειδιά ή μοτιβα
    if pd.isna(text): return 'Λοιποί Δικαιούχοι' #Αν το κείμενο είναι κενό, επιστρέφει τη προεπιλεγμένη τιμή 'Λοιποι Δικαιούχοι'
    text_upper = str(text).upper() #Μετατρέπουμε όλο το κείμενο σε κεφαλαία γράμματα για να είναι πιο εύκολή και ομοιόμορφη η αναζήτηση
    #Ψάχνουμε αν υπάρχουν συγκεκριμένα ονόματα και επιστ΄ρέφει το επίσημο όνομα του οργανισμού
    if 'ΜΕΤΑΔΡΑΣΗ' in text_upper or 'METADRASI' in text_upper: return 'ΜΕΤΑΔΡΑΣΗ'
    if 'ΑΡΣΙΣ' in text_upper or 'ARSIS' in text_upper: return 'ΑΡΣΙΣ'
    if 'PRAKSIS' in text_upper or 'ΠΡΑΞΙΣ' in text_upper: return 'PRAKSIS'
    if 'ΕΛΛΗΝΙΚΟΣ ΕΡΥΘΡΟΣ ΣΤΑΥΡΟΣ' in text_upper or 'ΕΕΣ' in text_upper: return 'ΕΛΛΗΝΙΚΟΣ ΕΡΥΘΡΟΣ ΣΤΑΥΡΟΣ'
    if 'SOLIDARITY' in text_upper or 'ΣΟΛΙΝΤΑΡΙΤΥ' in text_upper: return 'SOLIDARITY NOW'
    if 'ΙΟΜ' in text_upper or 'IOM' in text_upper or 'ΜΕΤΑΝΑΣΤΕΥΣΗ' in text_upper: return 'ΔΙΕΘΝΗΣ ΟΡΓΑΝΙΣΜΟΣ ΜΕΤΑΝΑΣΤΕΥΣΗΣ (ΔΟΜ)'
    
    match = re.search(r'([Α-ΩΆΈΉΊΌΎΏA-Z\s]{4,})\s+(?:Α\.Ε\.|Ε\.Π\.Ε\.|Ι\.Κ\.Ε\.|ΛΤΔ|LTD)', text_upper) #Αν δεν βρούμε καμί από τις παραπάνω γνωστές οργανώσεις, ψάχνει με regex για εταιρικές μορφές. Ψάχνουμε μια λέξη με τουλάχιστον 4 χαρακτήρες που ακολουθείται από συντομογραφίες 
    if match: #Αν βρει μια τέτοια λέξη
        return match.group(1).strip()#επιστρέφει το όνομα της καθαρισμένο απο κενά
        
    return 'Λοιποί Ανάδοχοι / Ιδιώτες' 

def extract_why(text): #Παίρνουμε το κείμενο και προσπαθούμε να εντοπίσουμε το σκοπό της δαπάνης
    if pd.isna(text): return 'Γενικές Δαπάνες' #Αν το κείμενο λείπει επιστρέφει 'Γενικές Δαπάνες΄'
    text_lower = str(text).lower() #Μετατρέπουμε το κείμενο σε μικρά γράμματα 
    #Ψάχνουμε αν υπάρχουν συγκεκριμένα ονόματα και επιστρέφουμε τον σκοπο της δαπάνης
    if 'σίτισ' in text_lower or 'φαγητ' in text_lower or 'γευμα' in text_lower: return 'Σίτιση / Τροφή'
    if 'στέγασ' in text_lower or 'διαμον' in text_lower or 'φιλοξεν' in text_lower or 'ενοίκι' in text_lower: return 'Στέγαση / Δομές'
    if 'καθαρι' in text_lower or 'απολύμαν' in text_lower or 'υγιειν' in text_lower: return 'Καθαριότητα / Υγιεινή'
    if 'φύλαξ' in text_lower or 'ασφάλει' in text_lower or 'security' in text_lower: return 'Φύλαξη / Ασφάλεια'
    if 'μεταφορ' in text_lower or 'εισιτηρ' in text_lower: return 'Μεταφορές / Μετακινήσεις'
    return 'Λοιπές Λειτουργικές Δαπάνες' #Αν δεν υπάρχουν όλα τα παραπάνω επιστρέφει 'Λοιπες Λειτουργικές Δαπάνες΄'

def extract_location(text): #Παίρνουμε το κείμενο και προσπαθούμε να εντοπισουμε τις περιοχές 
    if pd.isna(text): return 'Γενική / Κεντρική Διοίκηση' #Αν δεν υπάρχει κείμενο θεωρούμε ότι αφορά 'Γενική/Κεντρική Διοίκηση΄'
    text_lower = str(text).lower() #Μετατρέπουμε το κείμενο σε πεζά γράμματα
    locations_dict = { #Ορίζουμε ένα λεξικό, αριστερα είανι το μοτίβο που ψάχνει στο κείμενο και δεη=ξια το καθαρό όνομα της τοποθεσίας
        'μυτιλήν|λεσβ': 'Λέσβος', 'σάμο': 'Σάμος', 'χίο': 'Χίος', 'κω': 'Κως', 'λέρο': 'Λέρος',
        'μαλακάσ': 'Μαλακάσα', 'διαβατ': 'Διαβατά', 'σχιστ': 'Σχιστό', 'ελευσίν': 'Ελευσίνα',
        'ριτσών': 'Ριτσώνη', 'θερμοπύλ': 'Θερμοπύλες', 'έβρο|φυλάκι': 'Έβρος'
    }
    for pattern, name in locations_dict.items(): #Κανουμε μια λούπα για να ελέγξει ένα-ένα τα μοτίβα του λεξικού
        if re.search(pattern, text_lower): return name #Αν βρούμε το μοτίβο μέσα στο κείμενο, επιστρέφει αμέσως το επίσημο όνομα της τοποθεσίας και σταματαει
    return 'Λοιπές Δομές / Επαρχία' #Αν όμως η λούπα τελειώσει και δεν βρει καμία γνωστή τοποθεσία απο το λεξικό, επιστρέφει 'Λοιπες Δομές/Επαρχία'

def extract_type(text): #Παίρνουμε το κείμενο και προσπαθούμε να εντοπίσουμε τον τρόπο ανάθεσης ή το είδος της δαπάνης
    if pd.isna(text): return 'Τακτικές Δαπάνες'  #Αν δεν υπάρχει κείμενο θεωρούμε ότι αφορά 'Τακτικές Δαπάνες΄'
    text_lower = str(text).lower() #Μετατρέπουμε το κείμενο σε πεζά γράμματα
    if 'απευθείας' in text_lower or 'απευθειας' in text_lower: return 'Απευθείας Ανάθεση' #Αν βρει τη λέξη απευθέιας επιστρέφει 'Απευθείας Ανάθεση'
    if 'διαγωνισμ' in text_lower or 'μειοδοτ' in text_lower: return 'Διαγωνισμός' #Αν βρεί λε΄ξεις που σχετίζονται με διαγωνισμό η μειοδοσία επιστρέφει 'Διαγωνισμος'
    return 'Επιχορήγηση / Λοιπές Αναθέσεις' #Αν δεν ισχύει τιποτα απο τα δυο,επιστρέφει 'Επιχορήγηση/Λοιπες Αναθέσεις'
#Παι=ίρνουμε ενα νεο df και δημιουργούμε νεες στήλες
df['amount_clean'] = df['subject'].apply(extract_amount)
df['from_whom'] = df['organization']  
df['into_what'] = df['subject'].apply(extract_beneficiary)
df['why'] = df['subject'].apply(extract_why)
df['locations'] = df['subject'].apply(extract_location)
df['typos_anatheris'] = df['subject'].apply(extract_type)

df['clean_date'] = pd.to_datetime(df['issueDate'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
df['year'] = df['clean_date'].dt.year
df['month'] = df['clean_date'].dt.to_period('M')
print(f"Στήλες που είναι πλέον έτοιμες: {['from_whom', 'into_what', 'why', 'amount_clean', 'locations', 'typos_anatheris']}")

Στήλες που είναι πλέον έτοιμες: ['from_whom', 'into_what', 'why', 'amount_clean', 'locations', 'typos_anatheris']


In [28]:
#Ποιοι είναι οι κορυφαίοι 5 διακαιούχοι που έλαβαν την μεγαλύτερη χρηματοδότηση;
top_orgs = df.groupby('organization')['amount_clean'].sum() #Ομαδοποιεί τα δεδομένα με βάση τη στήλη του φορέα organization και αθροίζει τα καθαρά ποσα amount_clean για τον καθένα, υπλογίζοντας τα συνολικά χρήματα που διαχειρίστηκε κ΄άθε οργανισμός
top5_orgs = top_orgs.sort_values(ascending=False).head(5) #Ταξινομούμε τους οργανισμούς σε φθίνουσα σειρά και εμφανίζουμε τους 5 πρώτους
print(" ΚΟΡΥΦΑΙΟΙ ΦΟΡΕΙΣ/ΟΡΓΑΝΙΣΜΟΙ (organization):") 
for i, (org, money) in enumerate(top5_orgs.items(), 1): # Κάνουμε μια λούπα για να διαβάσει τους 5 κορυφαιους φορε΄ις και τους αριθμούμε με τη συναρτηση enumerate απο το 1 εως 5
    print(f"{i}. {org}:{money:,.2f}€") #Εμφανίζουμε το όνομα του οργανισμου και το συνολικό ποσό μορφοποιημένο με δυο δεκαδικά ψηφία 

 ΚΟΡΥΦΑΙΟΙ ΦΟΡΕΙΣ/ΟΡΓΑΝΙΣΜΟΙ (organization):
1. ΥΠΟΥΡΓΕΙΟ  ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ:195,735,186.34€
2. ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ:183,803,872.59€
3. ΥΠΟΥΡΓΕΙΟ ΕΡΓΑΣΙΑΣ ΚΑΙ ΚΟΙΝΩΝΙΚΗΣ ΑΣΦΑΛΙΣΗΣ:0.00€


In [29]:
#Πόσο συχνά επαναλαμβάνεται ο ΄ίδιος δικαιούχος στις αποφάσεις πληρομών;
συχνότητα_δικαιούχων = df['into_what'].value_counts() # Εξετάζουμε τη στήλη των δικαιούχων (into_what) και μετράμε πόσες φορές εμφανίζεται το όνομα κάθε δικαιούχου. Ταξινομούμε αυτόματα το αποτ΄έλεσμα από τη μεγαλύτερη συχνότητα προς τη μικρότερη
top5_εμφανίσεις = συχνότητα_δικαιούχων.head(5) #Εμφανίζουμε τους 5 δικαιούχους που εμφανίζονται πιο συχνα
print("Συχνότητα εμφάνισης των ίδιων δικαιούχων (Top 5):")
for όνομα, φορές in top5_εμφανίσεις.items(): #Κάνουμε μια λούπα γι αν διαβάσει τους 5 πρω΄τους δικαιούχους και σε κέθε επανάληψη η μεταβλητη όνομα παίρνει το όνομα του δικαιούχου και η μεταβλητή φορέασ το πλήθος των αποφάσεων  στις οποίες εμπλέκεται
    print(f" {όνομα}: εμφανίζεται {φορές} φορές στις αποφάσεις")


Συχνότητα εμφάνισης των ίδιων δικαιούχων (Top 5):
 Λοιποί Ανάδοχοι / Ιδιώτες: εμφανίζεται 7522 φορές στις αποφάσεις
 ΔΙΕΘΝΗΣ ΟΡΓΑΝΙΣΜΟΣ ΜΕΤΑΝΑΣΤΕΥΣΗΣ (ΔΟΜ): εμφανίζεται 503 φορές στις αποφάσεις
 ΑΡΣΙΣ: εμφανίζεται 50 φορές στις αποφάσεις
 ΜΕΤΑΔΡΑΣΗ: εμφανίζεται 49 φορές στις αποφάσεις
 PRAKSIS: εμφανίζεται 21 φορές στις αποφάσεις


In [30]:
# Ποιο είναι το μέσο κόστος άνα απόφαση όταν αφορά σίτιση σε σχέση με όταν αφορά στέγαση;
sitisi_df = df[df['why'].astype(str).str.contains('σίτισ|σιτιζ|γευμα|τροφίμ', case=False, na=False)] #Φιλτράρουμε το αρχικό df ψάχνοντας στη στήλη της αιτιολογίας why και κρατάμε μόνο τις αποφάσες που περιέχουν λέξεις σχετικές με φαγητό και τις αποθηκευουμε στο sitisi_df
stegasi_df = df[df['why'].astype(str).str.contains('στέγασ|φιλοξεν|διαμον|μίσθωσ', case=False, na=False)] #Φιλτράρουμε το αρχικό df ψάχνοντας στη στήλη της αιτιολογίας why και κρατάμε μόνο τις αποφάσεις που περιέχουν λέξεις σχετικές με στέγαση και τις αποθηκεύουμε στο stegasi_df
print("Μέσο κόστος ανά απόφαση:")
print(f" Σίτιση:  {sitisi_df['amount_clean'].mean():,.2f} € (Βάσει {len(sitisi_df)} αποφασέων)") #Υπολογίζουμε το μέσο όρο των ποσών στη στήλη amount_clean για τις αποφάσεις σίτισης
print(f" Στέγαση: {stegasi_df['amount_clean'].mean():,.2f} € (Βάσει {len(stegasi_df)} αποφασέων)") #Υπολογίζουμε το μέσο ΄όρο των ποσών στη στήλη amount_clean για  τις αποφάσεςι στέγασης 


Μέσο κόστος ανά απόφαση:
 Σίτιση:  189.19 € (Βάσει 37 αποφασέων)
 Στέγαση: 35,364.20 € (Βάσει 1655 αποφασέων)


In [31]:
#Ποια είναι η μεγαλύτερη εφαπαξ απόφαση; Ποιος είναι ο δικαιούχος και ποιος είναι ο φορέας που την όρισε;
if df['amount_clean'].notna().any(): #Ελέγχουμε ότι η στήλη τωνν ποσών περιέχει τουλαχιστον μια έγκυρη τιμή πριν προχωρήσει αλλιώς πηγαίνει στο else
    max_index = df['amount_clean'].idxmax() #Με την εντολή idxmax βρίσκουμε ποια ακριβώς γραμμή του πίνακα έχει την απόλυτη μέγιστη ιμή στη στήλη των ποσών amount_clean
    max_row = df.loc[max_index] #Χρησιμοποιώντας την εντολή max_index, απομονώνουμε ολόκληρη τη συγκεκριμένη γραμμλη με όλα τα στοιχεία της και την αποθηκεύουμε στη μεταβλητή max_row
    print("Η μεγαλύτερη μεμονωμένη απόφαση στη βάση δεδομένων:")
    print(f" Ποσό: {max_row['amount_clean']:,.2f} €") #Εξάγουμε το μέγιστο ποσό αποο τη γραμμή
    print(f" Φορέας (Από ποιον): {max_row['from_whom']}") #Τυπώνουμε το όνομα του φορέα που πλήρωσε την απόφαση
    print(f" Δικαιούχος (Σε ποιον): {max_row['into_what']}") #Τυπώνουμε το όνομα του δικαιούχου που έλαβε το ποσό
    print(f" Αιτιολογία: {max_row['why']}") #Τυπώνουμε το λόγο της δαπάνης
    print(f" ΑΔΑ: {max_row['ada']}") #Τυπώνουμε το αδα
else: #Αν η στήλη των ποσών ΄΄ήταν άδεια, θα τυπώνε το παρακάτω μηνυμα
    print("Δεν βρέθηκαν έγκυρα ποσά για τον υπολογισμό.")

Η μεγαλύτερη μεμονωμένη απόφαση στη βάση δεδομένων:
 Ποσό: 107,000,000.00 €
 Φορέας (Από ποιον): ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ
 Δικαιούχος (Σε ποιον): Λοιποί Ανάδοχοι / Ιδιώτες
 Αιτιολογία: Λοιπές Λειτουργικές Δαπάνες
 ΑΔΑ: 6ΥΞΤ46ΜΔΨΟ-Ψ1Ω


In [32]:
#Σε ποιές περιοχές ή δομές κατευθύνεται ο μεγαλύτερος όγκος χρημάτων;
top_locations = df.groupby('locations')['amount_clean'].sum() #Ομαδοποιεί τα δεδομένα με βάση τη τοποθεσία location και στη συνέχεια προσ΄θέτει όλα τα ποσά της στήλης amount_clean για κάθε τοποθε΄σια ξεχωριστα, υπολογίζοντας το συνολικό ποσό που απορρόφησε η καθεμία      
top5_locations = top_locations.sort_values(ascending=False).head(5) # Ταξινομούμε τις τοποθεσίες σε φθίνουσα σειρά και κρατάμε μόνο τις 5 πρώτες
print("Οι 5 κορυφαίες περιοχές/δομές σε απορρόληση χρημάτων:")
for i, (τοποθεσία, λεφτά) in enumerate(top5_locations.items(), 1): #Κάνουμε μια λούπα για να διαβάσει τις 5 κορυφαίες περιοχές και με τη συνάρτηση enumerate αριθμούμε από το 1 έως το 5. Σε κάθε βήμα, η μεταβλητη τοποθεσία παίρνει το όνομα της περιοχής και η μεταβλητη λεφτά το αντ΄ίστοιχο συνολικό ποσό
    print(f"{i}. {τοποθεσία}: {λεφτά:,.2f} €")

Οι 5 κορυφαίες περιοχές/δομές σε απορρόληση χρημάτων:
1. Λοιπές Δομές / Επαρχία: 241,520,476.62 €
2. Σάμος: 114,638,005.32 €
3. Λέσβος: 16,499,389.98 €
4. Κως: 4,098,469.99 €
5. Χίος: 1,684,568.37 €


In [44]:
#Υπάρχει διαφορά στη κατανομή των δαπάνων μεταξυ αστικών κέντρων και επαρχίας ή νησιών;
astika_kentra_keywords = 'αθήν|αττικ|θεσσαλον|κεντρική μακεδον|πειραι' #Ορίζουμε ένα string με λέξεις κλειδια που αντιπροσωπεύουν τα αστικά κέντρα της Ελλάδας
df_astika = df[df['locations'].astype(str).str.contains(astika_kentra_keywords, case=False, na=False)] #Φιλτράρουμε το αρχικό df και κρατάμε ΄μόνο τις γραμμές όπυ η στήλη locations περιέχει κάποια από τις λέξεις κλειδιά των αστικών κέντρων. Το αποτέλεσμα αποθηκεύεται δτο df_astika
df_eparchia = df[~df['locations'].astype(str).str.contains(astika_kentra_keywords, case=False, na=False)] # Φιλτράρουμε το df αλλα με την αντίθετη λογική γι αυτο χρησιμοποιούμε το συμβολο (~), δηλαδή κρατάμε όλες τις γραμμές που δεν περιέχουν καμία απο τις λέξεις κλεδια των αστικών κέντρων, απομομώνταας έτσι την Επαρχία, τα Νησιά και λοιπές δομές στο df_eparchia
poso_astika = df_astika['amount_clean'].sum() #Αθροίζουμε όλα τα ποσά των αποφάσεων που αφορούν αποκλειστικά τα μεγάλα αστικά κέντρα
poso_eparchia = df_eparchia['amount_clean'].sum() #Αθροίζουμε όλα τα ποσά των αποφάσεων που αφορούν την επαρχία, τα νησιά και τις υπολοιπες περιοχές
print("Κατανομή δαπανών: Αστικά Κέντρα vs Επαρχία/Νησιά:")
print(f" Μεγάλα Αστικά Κέντρα: {poso_astika:,.2f} € (Από {len(df_astika)} αποφάσεις)") # Εμφανίζουμε το συνολικό ποσό για τα αστικά κέντρα και χρησιμοποιώντας τη συνάρτηση len, τυπώνουμε και το πλήθος των αποφάσεων που οδ΄ήγησαν σε αυτό το ποσό
print(f" Επαρχία / Νησιά / Λοιπά: {poso_eparchia:,.2f} € (Από {len(df_eparchia)} αποφάσεις)") #Εμφανίζουμε το συνολικό ποσό για τα νησια και την επαρχία και χρησιμοποιώντας τη συνάρτηση len, τυπώνουμε και το πλήθος των αποφάσεων που οδήγησαν σε αυτο το ποσό


Κατανομή δαπανών: Αστικά Κέντρα vs Επαρχία/Νησιά:
 Μεγάλα Αστικά Κέντρα: 0.00 € (Από 0 αποφάσεις)
 Επαρχία / Νησιά / Λοιπά: 379,539,058.93 € (Από 8225 αποφάσεις)


In [34]:
#Ποιο μήνας του έτους παρουσιάζει τη μεγαλύτερη έκρηξη στη έκδοση αποφάσεων;
if 'month' in df.columns and df['month'].notna().any(): #Επαληθεύουμε αν η στήλη month υπάρχει στο df και αν περιέχει τουλάχιστον μια μη κενή τιμή . Αν η συνθήκη είναι αληθής, προχωράει στον κώδικα αλλιώς οδηγείτε στο else
    αποφάσεις_ανά_μήνα = df['month'].value_counts() # Μετράμε πόσες φορές εμφανίζεται η στήλη month και τη ταξινομουμε σε φθίνουσα σειρά
    top3_months = αποφάσεις_ανά_μήνα.head(3) #Κρατάμε μόνο τους 3 πρώτους μήνες
    print("Οι 3 μήνες με τη μεγαλύτερη έκρηξη σε πλήθος αποφάσεων:")
    for i, (μήνας, πλήθος) in enumerate(top3_months.items(), 1): #Κάνουμε μια λούπα για να διαβάσει τους 3 κορυφαίους μήνες και με την συνάρτηση enumerate αριθμούμε απο το 1 εως το 3. Σε κάθε βήμα, η μεταβλητη μήνας λαμβάνει τη τιμή του μήνα και η μεταβλητη πλήθος τον αριθμό των αποφάσεων 
        print(f"{i}. Μήνας: {μήνας} -> {πλήθος} αποφάσεις")
else: #Στη περίπτωση που ο αρχικός έλεγχος αποτύχει, τυπώνουμε το παρακάτω μήνυμα
    print(" Δεν βρέθηκαν έγκυρα δεδομένα ημερομηνίας (στήλη 'month') στο αρχείο σου.")

Οι 3 μήνες με τη μεγαλύτερη έκρηξη σε πλήθος αποφάσεων:
1. Μήνας: 2020-12 -> 263 αποφάσεις
2. Μήνας: 2020-09 -> 165 αποφάσεις
3. Μήνας: 2021-03 -> 160 αποφάσεις


In [35]:
#Υπάρχει ασυνήθιστη αυξητική τάση αποφάσεων (πχ πριν τις εκκλογές);
if 'month' in df.columns and df['month'].notna().any(): #Ελέγχουμε αν υπάρχει η στηλη month και αν υπάρχει έστω μια έγκυρη ημερομηνια σε αυτή τη στήλη. Αν η στήλη είναι άδεια, προσπερνάει τον κωδικα για να μην βγάλει σφάλμα 
    μηνιαία_τάση = df.groupby('month')['amount_clean'].sum()#Ομαδοποιούμε όλες τις εγγραφές ανα μήνα και αθροίζουμε τα καθαρά ποσα, για να βρούμε το συνολικό ποσό που δαπανήθηκε για κάθε μήνα ξεχωριστα
    print("Μηνιαία εξέλιξη των δαπανών (Τάση):")
    for μήνας, ποσό in μηνιαία_τάση.items(): #Κάνουμε μια λούπα γι αν διαβάσουμε έναν-έναν τους μήνες και τα αντιστοιχα ποσά
        print(f" Μήνας {μήνας}: {ποσό:,.2f} €")#Τυπώνουμε το αποτέλεσμα στην οθόνη με ωραία μορφοποιήση βάζοντας υποδιαστολή στα δυο δεκαδικά ψηφία
else: #Αλλιώς τυπώνουμε τη παρακάτω εντολή
    print(" Δεν βρέθηκαν έγκυρα δεδομένα ημερομηνίας (στήλη 'month') στο αρχείο σου.")

Μηνιαία εξέλιξη των δαπανών (Τάση):
 Μήνας 2020-01: 0.00 €
 Μήνας 2020-02: 196,208.92 €
 Μήνας 2020-03: 4,103,818.90 €
 Μήνας 2020-04: 1,237,235.85 €
 Μήνας 2020-05: 2,134,103.60 €
 Μήνας 2020-06: 256,163.19 €
 Μήνας 2020-07: 97,124.92 €
 Μήνας 2020-08: 112,635,562.60 €
 Μήνας 2020-09: 157,458.77 €
 Μήνας 2020-10: 488,987.42 €
 Μήνας 2020-11: 2,813,025.66 €
 Μήνας 2020-12: 10,564,479.49 €
 Μήνας 2021-01: 49,811.18 €
 Μήνας 2021-02: 4,694,785.42 €
 Μήνας 2021-03: 4,845,330.29 €
 Μήνας 2021-04: 408,449.94 €
 Μήνας 2021-05: 178,296.39 €
 Μήνας 2021-06: 620,852.14 €
 Μήνας 2021-07: 4,726,014.99 €
 Μήνας 2021-08: 2,648,323.86 €
 Μήνας 2021-09: 271,972.19 €
 Μήνας 2021-10: 2,015,829.32 €
 Μήνας 2021-11: 4,134,756.65 €
 Μήνας 2021-12: 12,021,629.61 €
 Μήνας 2022-01: 607,395.25 €
 Μήνας 2022-02: 5,831,218.01 €
 Μήνας 2022-03: 167,340.20 €
 Μήνας 2022-04: 4,193,651.60 €
 Μήνας 2022-05: 1,022,957.44 €
 Μήνας 2022-06: 4,471,261.10 €
 Μήνας 2022-07: 1,985,885.88 €
 Μήνας 2022-08: 2,210,846.11 €
 Μ

In [36]:
#Ποιοι είναι οι πιο συχνοι τύποι αποφάσεων;
df['typos_anatheris'] = 'Λοιπές / Τακτικές δαπάνες'
#Τσεκα΄ρουμε τη στήλη why για να επαναπροσδιορίσουμε τον τύπο ανάθεσης, χρησιμοποιώντας τη μέθοδο .str.contains()
df.loc[df['why'].astype(str).str.contains('απευθείας|απευθειας', case=False, na=False), 'typos_anatheris'] = 'Απευθείας Ανάθεση' 
df.loc[df['why'].astype(str).str.contains('διαγωνισμ|μειοδοτ', case=False, na=False), 'typos_anatheris'] = 'Διαγωνισμός'
df.loc[df['why'].astype(str).str.contains('επιχορήγησ|επιχορηγησ|χρηματοδότησ', case=False, na=False), 'typos_anatheris'] = 'Επιχορήγηση / Έκτακτη Δαπάνη'
print("Κατανομή ανά τύπο απόφασης/ανάθεσης:")
print(df['typos_anatheris'].value_counts())

Κατανομή ανά τύπο απόφασης/ανάθεσης:
typos_anatheris
Λοιπές / Τακτικές δαπάνες    8225
Name: count, dtype: int64


In [37]:
φορείς_υπογραφής = df['from_whom'].value_counts() #Μετράμε πόσε φορές εμφανίστηκε ο κάθε φορέας απο τη στήλη from_whom
top5_φορείς = φορείς_υπογραφής.head(5)# Κρατάμε μόνο τους 5 πρώτους φορέις απο τη λίστα 
print("Οι 5 φορείς/υπηρεσίες που υπογράφουν τις περισσότερες δαπάνες:")
for i, (φορέας, πλήθος) in enumerate(top5_φορείς.items(), 1): #Κάνουμε μια λούπα για να εμφανίσει έναν-έναν τους 5 φορείς, με το enumerate δημιουργούμε μια αρίθμηση που ξεκινάει απο το 1. Σε κάθε επανάληψη η μεταβήτη φορέας παίρνει το όνομα της υπηρεσίας και η μεταβλητη πλήθος το αριθμό των αποφάσεων
    print(f"{i}. {φορέας} -> Έχει υπογράψει {πλήθος} αποφάσεις") #Τυπώνουμε το αποτέλεσμα μορφοποιημένα

Οι 5 φορείς/υπηρεσίες που υπογράφουν τις περισσότερες δαπάνες:
1. ΥΠΟΥΡΓΕΙΟ  ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ -> Έχει υπογράψει 5017 αποφάσεις
2. ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ -> Έχει υπογράψει 3205 αποφάσεις
3. ΥΠΟΥΡΓΕΙΟ ΕΡΓΑΣΙΑΣ ΚΑΙ ΚΟΙΝΩΝΙΚΗΣ ΑΣΦΑΛΙΣΗΣ -> Έχει υπογράψει 3 αποφάσεις


In [38]:
#Σε ποιο έτος παρατηρήθηκε η μεγαλύτερη δαπάνη; 
if 'year' in df.columns and df['year'].notna().any(): #Ελέγχουμε αν υπάρχει η στήλη year στο df και αν αυτή η στήλη περιέχει έστω και μια έγκυρη τιμή. Αν ισχύει εκτελείτε ο κώδικα
    ετήσιες_δαπάνες = df.groupby('year')['amount_clean'].sum()# Ομαδοποιεί τα δεδομένα ανά έτος (groupby(year)) και προσθέτει όλα τα ποσά απο τη στήλη amount_clean για κάθε έτος, υπλογίζοντας έτσι το συνολοκό κόστος ανά χρονιά
    top_years = ετήσιες_δαπάνες.sort_values(ascending=False) #Ταξινομούμε τα έτη σε φθίνουσα σειρά με βάση το συνολικό τους ποσό 
    print("Κατάταξη των ετών με βάση τη συνολική δαπάνη:")
    for έτος, ποσό in top_years.items(): #Κάνουμε μια λούπα για να διαβάσουμε κάθε έτος και το αντίστοιχο ποσό που δαπανήθηκε
        print(f" Έτος {int(έτος)}: {ποσό:,.2f} €") #Τυπώνουμε το έτος, μετατρεποντάς το σε ακέραιο int και δίπλα το ποσό μορφοποιημένο με κομμα για τις χιλιάδες και 2 δεκαδικά
else: # Αν ο αρχικός έλεγχος αποτύχει, τυπώνουμε το παρακάτω μηνυμα
    print("Δεν βρέθηκαν έγκυρα δεδομένα έτους (στήλη 'year') στο αρχείο σου.") 

Κατάταξη των ετών με βάση τη συνολική δαπάνη:
 Έτος 2020: 134,684,169.32 €
 Έτος 2025: 61,197,556.81 €
 Έτος 2024: 45,251,552.97 €
 Έτος 2022: 36,649,152.93 €
 Έτος 2021: 36,616,051.98 €
 Έτος 2023: 34,002,948.94 €
 Έτος 2026: 31,137,625.98 €


In [39]:
direct_df = df[df['typos_anatheris'] == 'Απευθείας Ανάθεση'] # Φιλτράρουμε το αρχικό df και κρατάμε μόνο τις γραμμές όπου η στήλη typos_anathesis έχει ακριβώς τη τιμή Απευθείασ Ανάθεση και αποθηκεύει το αποτέλεσμα σε ενα νεό df με όνομα direct_df 
top_direct_beneficiaries = direct_df.groupby('into_what')['amount_clean'].sum() #Ομαδοποιούμε τα δεδομένα με βάση τον δικαιούχο (στηλη into_what) και για κέθε δικαιούχο προσθέτει όλα τα ποσά (στήλη amount_clean) ώστε να βγει το συνολοκό άθροισμα που έλαβε
top5_direct = top_direct_beneficiaries.sort_values(ascending=False).head(5) #Ταξινομούμε τους δικαιούχος σε φθίνουσα σειρά και κρατάμε μόνο τους 5 πρώτους
print("Οι Top 5 δικαιούχοι ΜΟΝΟ από Απευθείας Αναθέσεις:") #Τυπώνουμε τους 5 πρώτους δικαιούχος
for i, (name, money) in enumerate(top5_direct.items(), 1): #Κάνουμε μια λούπα για να διαβάσει έναν-έναν τους 5 κορυφαίους δικαιούχους
    print(f"{i}. {name}: {money:,.2f} €")


Οι Top 5 δικαιούχοι ΜΟΝΟ από Απευθείας Αναθέσεις:


In [24]:
network_funding = df.groupby(['from_whom', 'into_what'])['amount_clean'].sum() # Ομαδοποιούμε το αρχικο΄ df με βάση δυο στήλες ταυτόχρονα: το φορέα που δίνει τα χρήματα (from_whom) και τον δικαιούχο που τα παίρνει (into_what). Στη συνέχεια αθροιζουμε τα ποσά για κάθε μοναδικό συνδυασμό φορέα-δικαιούχου
top5_pairs = network_funding.sort_values(ascending=False).head(5) #Ταξινομούμε τα ζευγάρια σε φθίνουσα σειρά με βάση το συνολικο ποσό και κρατάμε μόνο τα 5 ζευγάρια που είχαν την μεγαλύτερη ροή χρήματος
print("Τα 5 ισχυρότερα 'ζευγάρια' (Φορέας ➔ Δικαιούχος) σε ροή χρήματος:")
for i, ((φορέας, δικαιούχος), ποσό) in enumerate(top5_pairs.items(), 1): #Ξεκινάμε μια λούπα για να διατρέξουμε στα 5 κορυφαία ζευγάρια, και επειδή η ομαδοποίηση έγινε με δυο στήλες το κλειδι περιέχει δυο στοιχεια (φορέας, δικαιούχος) και η τιμή είναι το ποσό
    print(f"{i}.  {φορέας} \n   ➔  {δικαιούχος} \n   Συνολικό Ποσό: {ποσό:,.2f} €") #Τυπώνουμε το αποτέλεσμα, το \n αλλάζει γραμμή, ωστε να φαίνεται καθαρά ο φορέας και απο κατω με βε΄λος ο δικαιούχος και απο κατω το ποσο με 2 μονο δεκαδικα
    print("-" * 50) #Τυπώνουμε μια διαχωριστική γραμμη με 50 παυλες ωστε να ειναι ευανάγνωστο 

Τα 5 ισχυρότερα 'ζευγάρια' (Φορέας ➔ Δικαιούχος) σε ροή χρήματος:
1.  ΥΠΟΥΡΓΕΙΟ  ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ 
   ➔  Λοιποί Ανάδοχοι / Ιδιώτες 
   Συνολικό Ποσό: 194,411,465.26 €
--------------------------------------------------
2.  ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ 
   ➔  Λοιποί Ανάδοχοι / Ιδιώτες 
   Συνολικό Ποσό: 181,242,378.94 €
--------------------------------------------------
3.  ΥΠΟΥΡΓΕΙΟ ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ 
   ➔  ΔΙΕΘΝΗΣ ΟΡΓΑΝΙΣΜΟΣ ΜΕΤΑΝΑΣΤΕΥΣΗΣ (ΔΟΜ) 
   Συνολικό Ποσό: 2,455,304.01 €
--------------------------------------------------
4.  ΥΠΟΥΡΓΕΙΟ  ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ 
   ➔  ΕΛΛΗΝΙΚΉ ΕΤΑΙΡΕΊΑ ΣΥΜΜΕΤΟΧΏΝ
ΚΑΙ ΠΕΡΙΟΥΣΊΑΣ 
   Συνολικό Ποσό: 701,840.00 €
--------------------------------------------------
5.  ΥΠΟΥΡΓΕΙΟ  ΜΕΤΑΝΑΣΤΕΥΣΗΣ ΚΑΙ ΑΣΥΛΟΥ 
   ➔  ΣΎΜΒΑΣΗΣ ΜΕΤΑΞΎ ΤΟΥ ΥΠΟΥΡΓΕΊΟΥ ΜΕΤΑΝΆΣΤΕΥΣΗΣ ΚΑΙ ΑΣΎΛΟΥ ΚΑΙ ΤΗΣ ΕΛΛΗΝΙΚΉΣ ΕΤΑΙΡΕΊΑΣ
ΣΥΜΜΕΤΟΧΏΝ ΚΑΙ ΠΕΡΙΟΥΣΊΑΣ 
   Συνολικό Ποσό: 561,472.00 €
--------------------------------------------------


In [40]:
#Ποιες ειναι οι 5 μεγαλύτερες ΜΚΟ σε συνολική χρηματοδότηση
mko_only = df[~df['into_what'].astype(str).str.contains('Ανάδοχοι|Ιδιώτες|Υπουργείο|ΕΕΣΥΠ', case=False, na=False)]

top_mko = mko_only.groupby('into_what')['amount_clean'].sum().sort_values(ascending=False).head(5)

print("Οι Top 5 ΜΚΟ σε συνολικό ποσό χρηματοδότησης:")
for mko, money in top_mko.items():
    print(f"- {mko}: {money:,.2f} €")

Οι Top 5 ΜΚΟ σε συνολικό ποσό χρηματοδότησης:
- ΔΙΕΘΝΗΣ ΟΡΓΑΝΙΣΜΟΣ ΜΕΤΑΝΑΣΤΕΥΣΗΣ (ΔΟΜ): 2,492,120.72 €
- ΕΛΛΗΝΙΚΉ ΕΤΑΙΡΕΊΑ ΣΥΜΜΕΤΟΧΏΝ
ΚΑΙ ΠΕΡΙΟΥΣΊΑΣ: 701,840.00 €
- ΠΡΟΓΡΑΜΜΑΤΙΚΉΣ ΣΥΜΦΩΝΊΑΣ ΜΕΤΑΞΎ ΤΗΣ ΓΕΝΙΚΉΣ ΓΡΑΜΜΑΤΕΊΑΣ ΚΑΙ ΤΗΣ ΚΟΙΝΩΝΊΑΣ ΤΗΣ ΠΛΗΡΟΦΟΡΊΑΣ: 59,000.00 €
- ΤΗΝ ΕΠΙΧΟΡΉΓΗΣΗ ΤΗΣ ΚΤΠ: 24,689.64 €
- ΕΛΤΑ: 22,500.00 €


In [41]:
#Ποιο είναι το μέσο ποσό ανά απόφαση για κάθε φορέα
avg_per_beneficiary = df.groupby('into_what')['amount_clean'].agg(['count', 'mean', 'sum']).sort_values(by='sum', ascending=False).head(5)

print("Μέσο κόστος ανά εγκεκριμένη απόφαση (Top 5):")
print(avg_per_beneficiary)


Μέσο κόστος ανά εγκεκριμένη απόφαση (Top 5):
                                                    count           mean  \
into_what                                                                  
Λοιποί Ανάδοχοι / Ιδιώτες                            7522   49940.686546   
ΔΙΕΘΝΗΣ ΟΡΓΑΝΙΣΜΟΣ ΜΕΤΑΝΑΣΤΕΥΣΗΣ (ΔΟΜ)                503    4954.514354   
ΕΛΛΗΝΙΚΉ ΕΤΑΙΡΕΊΑ ΣΥΜΜΕΤΟΧΏΝ\nΚΑΙ ΠΕΡΙΟΥΣΊΑΣ            1  701840.000000   
ΣΎΜΒΑΣΗΣ ΜΕΤΑΞΎ ΤΟΥ ΥΠΟΥΡΓΕΊΟΥ ΜΕΤΑΝΆΣΤΕΥΣΗΣ ΚΑ...      1  561472.000000   
ΠΡΟΓΡΑΜΜΑΤΙΚΉΣ ΣΥΜΦΩΝΊΑΣ ΜΕΤΑΞΎ ΤΗΣ ΓΕΝΙΚΉΣ ΓΡΑ...      1   59000.000000   

                                                             sum  
into_what                                                         
Λοιποί Ανάδοχοι / Ιδιώτες                           3.756538e+08  
ΔΙΕΘΝΗΣ ΟΡΓΑΝΙΣΜΟΣ ΜΕΤΑΝΑΣΤΕΥΣΗΣ (ΔΟΜ)              2.492121e+06  
ΕΛΛΗΝΙΚΉ ΕΤΑΙΡΕΊΑ ΣΥΜΜΕΤΟΧΏΝ\nΚΑΙ ΠΕΡΙΟΥΣΊΑΣ        7.018400e+05  
ΣΎΜΒΑΣΗΣ ΜΕΤΑΞΎ ΤΟΥ ΥΠΟΥΡΓΕΊΟΥ ΜΕΤΑΝΆΣΤΕΥΣΗΣ ΚΑ...  5.614720e+05  
ΠΡΟΓΡΑΜΜΑΤΙΚΉΣ ΣΥΜΦΩ